# 实战-多功能智能助手

- 项目需求：开发一个多功能智能助手，支持:
1. **天气查询**：查询城市天气
2. **数学计算**：复杂数学运算
3. **时间查询**：查询当前时间、日期转换
4. **货币转换**：多种货币之间转换
5. **信息搜索**：搜索产品、新闻等信息

In [ ]:
import math

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

In [ ]:
class GetWeatherSchema(BaseModel):
    city: str = Field(min_length=2,max_length=10,description="城市名")

class CalculatorSchema(BaseModel):
    expression:str = Field(
        description="数学表达式"
    )

In [ ]:
@tool(args_schema=GetWeatherSchema,description="查询指定城市天气")
async def get_weather(city: str) -> str:
    weather_db = {
        "北京":"多云，10℃-20℃ 空气质量良好 湿度40%",
        "上海":"晴朗，21℃-30℃ 空气质量优秀 湿度70%",
        "深圳":"小雨，19℃-28℃ 空气质量良好 湿度97%",
        "广州":"多云，19℃-30℃ 空气质量良好 湿度80%",
        "杭州":"雷阵雨，20℃-27℃ 空气质量良好 湿度90%",
    }
    result = weather_db.get(city)
    if result:
        return f"{city}:{result}"
    else:
        return f"暂不支持查询{city}的天气,当前支持[北京,上海,广州,深圳,杭州]"

@tool(args_schema=CalculatorSchema,description="执行数学运算")
async def calculator(expression:str) -> str:
    try:
        safe_functions = {
            "sqrt":math.sqrt,
            "pow":pow,
            "abs":abs,
            "round":round,
            "sin":math.sin,
            "cos":math.cos,
            "tan":math.tan,
            "log":math.log,
            "pi":math.pi,
            "e":math.e,
        }

        result = eval(expression,{"__builtins__":{}},safe_functions)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算出错: {str(e)}\n提示：请检查表达式格式，支持的运算有：[sqrt,pow,abs,round,sin,cos,tan,log,pi,e]"

In [ ]:
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={
        "thinking":{"type":"disabled"}
    }
)

async_agent = create_agent(
    name="多功能智能助手",
    model=model,
    system_prompt=ToolStrategy(),
    tools=[]
)

async for chunk in async_agent.astream(
        {
            "messages":[
                HumanMessage(content="")
            ]
        }
):
    pass